# JAX FDM workshop — Google Colab

A browser fallback for the [JAX FDM workshop](https://github.com/arpastrana/jaxfdm-workshop).

**Runtime → Run all**, or run the cells in order from the top. Use `%run` for the `.py` examples, not `!python`: a subprocess cannot draw the notebook 3D viewer.

This notebook installs JAX FDM for Colab's notebook viewer. It does not install the desktop Qt viewer.

## 1. Check the Python version

Colab should report Python 3.12 or 3.13. Both work with JAX FDM 0.14.1.

In [ ]:
import sys

print(sys.version)

## 2. Install JAX FDM

This is the workshop package plus the notebook 3D viewer (`compas_notebook` and `pythreejs`). The first run takes a minute or two.

In [ ]:
%pip install -q "jax-fdm==0.14.1" "compas_notebook>=0.11" "pythreejs>=2.1" "compas_plotter>=1.0.1"

## 3. Confirm the install

A tiny four-bar solve. If this cell prints a version and an equilibrium point, JAX FDM is ready.

In [ ]:
import jax_fdm
from jax_fdm.datastructures import FDNetwork
from jax_fdm.equilibrium import fdm

print("jax-fdm", jax_fdm.__version__)

network = FDNetwork()
free_node = network.add_node(x=0.0, y=0.0, z=0.0)

supports = [
    [-1.0, 0.0, 0.0],
    [1.0, 0.0, 0.0],
    [0.0, 1.0, 0.5],
    [0.0, -1.0, 0.5],
]

for xyz in supports:
    node = network.add_node(x=xyz[0], y=xyz[1], z=xyz[2])
    network.node_support(node)
    network.add_edge(free_node, node)

network.node_load(free_node, [0.0, 0.0, -1.0])

for edge in network.edges():
    network.edge_forcedensity(edge, -1.0)

eq_network = fdm(network)
eq_xyz = eq_network.node_coordinates(free_node)
print("equilibrium xyz", eq_xyz)

## 4. Get the workshop files

Clones the repository into `/content` if it is not already there.

In [ ]:
from pathlib import Path

REPO_DIR = Path("/content/jaxfdm-workshop")

if REPO_DIR.exists():
    print("Workshop repository is already cloned.")
else:
    %cd /content
    !git clone --depth 1 https://github.com/arpastrana/jaxfdm-workshop.git

## 5. Point the 3D viewer at the notebook backend

The workshop scripts import `Viewer`, which opens a desktop window. This cell replaces it with `NotebookViewer` so the same `viewer.show()` calls draw a widget here. Run this once before any example.

In [ ]:
from google.colab import output
from jax_fdm.visualization import NotebookViewer
import jax_fdm.visualization as visualization

output.enable_custom_widget_manager()
visualization.Viewer = NotebookViewer

print("Viewer is now NotebookViewer")

## 6. Four bars — `01_form_finding/four_bars.py`

Prescribed forces take several Newton steps. Force densities take one. A convergence chart and a 3D widget follow the printed traces.

In [ ]:
%cd /content/jaxfdm-workshop/01_form_finding
%run four_bars.py

## 7. Gridshell — `03_gridshell/gridshell.py`

Constrained form-finding: stay close to a target shape, flatten the panels, keep the mesh even. The optimizer may take a minute. The widget colors each panel by how far it is from flat.

In [ ]:
%cd /content/jaxfdm-workshop/03_gridshell
%run gridshell.py

## 8. Other examples

Uncomment one pair and run the cell. Change back into the exercise folder first so local helpers import.

In [ ]:
# %cd /content/jaxfdm-workshop/01_form_finding
# %run four_bars_jaxfdm.py

# %cd /content/jaxfdm-workshop/02_arches
# %run arches.py
# %run arches_constrained.py

# %cd /content/jaxfdm-workshop/04_cablenet
# %run cablenet.py
# %run cablenet_constrained.py